# Manifold learning (_Moon_)

In [ ]:
from experiments.utils.constants import RANDOM_SEED, SOM_LEARNING_RATE_DECAY_FN, SOM_FIT_METHOD

VERBOSE = True
MANIFOLD_NAME = "Moon"
MANIFOLD_NAME_SAFE = MANIFOLD_NAME.lower().replace(" ", "_").replace("-", "")
MODELING_MODE = False
EXPORT_MODE = False
EXPORT_DIR = "_exports"

from experiments.utils.data_manifold import generate_moon
MANIFOLD_GENERATOR = generate_moon

print(MANIFOLD_NAME, MANIFOLD_NAME_SAFE)

## Dataset

In [ ]:
# generate data
from experiments.utils.plotting import visualize_manifold
X, y = MANIFOLD_GENERATOR()

# visualize 1/2
import matplotlib.pyplot as plt
figMan1 = plt.figure(figsize=(10, 10))
ax0 = figMan1.add_subplot(121, projection="3d")
visualize_manifold(X, y, title=f"{MANIFOLD_NAME} manifold in ambient space", ax=ax0, hold_on=True);

# scale data
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# visualize 2/2
ax1 = figMan1.add_subplot(122, projection="3d")
visualize_manifold(X_scaled, y, title=f"{MANIFOLD_NAME} manifold (scaled) in ambient space", ax=ax1, hold_on=False);

## Modeling

In [ ]:
from minisom_representation import calc_som_hyparams, SomRepresentation, plot_som_convergence_over_epochs

In [ ]:
# use helper methods to get SOM hyperparameter recommendations
recommended_params = calc_som_hyparams(X_scaled, initial_sigma_factor=3.0)
print("Recommended SOM parameters:", recommended_params)

In [ ]:
# define actual hyperparameters
d1, d2, sigma = map(recommended_params.get, ("d1", "d2", "sigma"))
decay_function = SOM_LEARNING_RATE_DECAY_FN
epoch = None

In [ ]:
# test candidate values for `num_iteration` hyperparameter
if MODELING_MODE:
    fig, errors_qe, errors_te = plot_som_convergence_over_epochs(
        SomRepresentation(d1=d1, d2=d2, sigma=sigma, random_seed=RANDOM_SEED, verbose=False, decay_function=decay_function),
        X_scaled,
        fit_type=SOM_FIT_METHOD, te_ceiling=.1,
        epoch_step_from=2, epoch_step_to=50, epoch_step=1,
        figsize=(16, 5), show_fig=True, verbose=VERBOSE
    )
    print(f"\nQE (first -> last): \t {errors_qe[0]:.2f} -> {errors_qe[-1]:.2f}")
    print(f"TE (first -> last): \t {errors_te[0]:.2f} -> {errors_te[-1]:.2f}")
else: print("Skipping model candidate parameters evaluation.")

In [ ]:
# set selected `num_iteration` as epoch
epoch = 28

In [ ]:
# fit SOM representation
som_rep = SomRepresentation(d1=d1, d2=d2, sigma=sigma, random_seed=RANDOM_SEED, verbose=VERBOSE, decay_function=decay_function) \
    .fit_online(X_scaled, num_iteration=epoch)

## Direct inspection of the fitted representation model

In [ ]:
# extract SOM prototype weights
som = som_rep.som
node_weights = som.get_weights()
node_weights_flat = node_weights.reshape(-1, node_weights.shape[2])

In [ ]:
from utils.plotting import place_som_lattice
data_plot_style = { "alpha": 0.45 }
figMan2, ax = visualize_manifold(X_scaled, y, f"SOM embedding of\n{MANIFOLD_NAME} manifold (scaled) in ambient space", plot_style_args=data_plot_style, hold_on=True)
place_som_lattice(ax, node_weights, node_weights_flat)

In [ ]:
if EXPORT_MODE:
    figMan1.savefig(EXPORT_DIR + "/01_01_lilypond_manifold_01.png")
    figMan2.savefig(EXPORT_DIR + "/01_01_lilypond_manifold_02.png")

## Inspection of the learned 2D topology

In [ ]:
from utils.plotting import PlotlyHelperArgs

In [ ]:
# create Basin
from lilypond import Basin
basin = Basin.from_som_representation(som_rep, random_seed=RANDOM_SEED, verbose=VERBOSE)

In [ ]:
# export basin
if EXPORT_MODE:
    from utils.export import BasinWithTrainingData
    BasinWithTrainingData(dataset_name=MANIFOLD_NAME, basin=basin, X_train=X_scaled).export(EXPORT_DIR + "/other")

In [ ]:
# traditional visuals
plot_args = dict(
    **PlotlyHelperArgs.FullStretch,
    **PlotlyHelperArgs.HiddenTicks(d1=d1, d2=d2),
    font=dict(size=35),
    title="",
)
figTrad1 = basin.legacy_pond().visualize_distance_map(**plot_args, **PlotlyHelperArgs.Figsize(w=950, h=600));
figTrad2 = basin.legacy_pond().visualize_activation_map(**plot_args, **PlotlyHelperArgs.Figsize(w=850, h=600));

In [ ]:
if EXPORT_MODE:
    figTrad1.write_image(EXPORT_DIR + "/01_01_lilypond_trad_01.png")
    figTrad2.write_image(EXPORT_DIR + "/01_01_lilypond_trad_02.png")

In [ ]:
# lilypond visual
basin.pond() \
    .rhizome_layer() \
    .pad_layer() \
    .petal_layer() \
    .visualize(width=800, height=600);

## Detailed inspection

In [ ]:
plot_args = dict(
    **PlotlyHelperArgs.Figsize(w=400, h=400),
    **PlotlyHelperArgs.FullStretch,
    **PlotlyHelperArgs.HiddenTicks(d1=d1, d2=d2),
    font=dict(size=35),
    showlegend=False
)

In [ ]:
fig1 = basin.pond() \
    .pad_layer() \
    .visualize(**plot_args);

In [ ]:
fig2 = basin.pond() \
    .pad_layer(gap="nogap") \
    .visualize(**plot_args);

In [ ]:
fig3 = basin.pond() \
    .pad_layer(gap="nogap") \
    .petal_layer() \
    .visualize(**plot_args);

In [ ]:
fig4 = basin.pond() \
    .pad_layer(gap="nogap") \
    .rhizome_layer() \
    .attraction_layer(X_scaled, marker=dict(color=y, colorscale="plasma", symbol="star-diamond", opacity=.8), name="Projection of training data colored by Manifold position") \
    .visualize(**plot_args);

In [ ]:
fig5 = basin.pond() \
    .pad_layer(gap="nogap") \
    .rhizome_layer(violations_only=True) \
    .attraction_layer(X_scaled, marker=dict(color=y, colorscale="plasma", symbol="star-diamond", opacity=.8), name="Projection of training data colored by Manifold position") \
    .visualize(**plot_args);

In [ ]:
if EXPORT_MODE:
    fig1.write_image(EXPORT_DIR + "/01_01_lilypond_01.png")
    fig2.write_image(EXPORT_DIR + "/01_01_lilypond_02.png")
    fig3.write_image(EXPORT_DIR + "/01_01_lilypond_03.png")
    fig4.write_image(EXPORT_DIR + "/01_01_lilypond_04.png")
    fig5.write_image(EXPORT_DIR + "/01_01_lilypond_05.png")

---

### The below cells are not part of the experiment. They are used to persist the data and register the model in Databricks and Bianor for further interactive investigation.

---

## Preparation

In [ ]:
MODEL_REGISTRATION_MODE = False
DATASET_PERSIST_MODE = False

In [ ]:
import pandas as pd
import mlflow

from dotenv import load_dotenv
from utils.databricks_util import get_spark, get_catalog_path, CATALOG, SCHEMA

In [ ]:
load_dotenv()
spark = get_spark()

## Persist data in Databricks

In [ ]:
TABLE_NAME = f"T_{MANIFOLD_NAME_SAFE}".lower()
TABLE_PATH = get_catalog_path(TABLE_NAME)
print(TABLE_PATH)

In [ ]:
if DATASET_PERSIST_MODE:
	# persist full dataset as a managed table
    spark.createDataFrame(
		pd.DataFrame(X) \
			.assign(manifold_position=y) \
			.reset_index(names="id")
	).write \
		.mode("overwrite") \
		.saveAsTable(TABLE_PATH)

In [ ]:
data_dbdf = spark.read \
    .table(TABLE_PATH)
data_dbdf.show(5)

In [ ]:
# separate variables
primary_key = ['id']
target = ['manifold_position']
data_df = data_dbdf.toPandas()
features = data_df.columns.difference((primary_key + target), sort=False).tolist()
assert primary_key[0] not in features, "Primary key shall not be in Features"
assert all(t not in features for t in target), "Targets must not be in Features"
print("Primary key:", primary_key)
print("Targets:", target)
print("Features:", features)

In [ ]:
# create feature dataframe
feature_dbdf = data_dbdf.select(primary_key + features)
feature_dbdf.show(5)

In [ ]:
# create feature table
from databricks.feature_engineering import FeatureEngineeringClient
FEATURE_TABLE_NAME = f"{TABLE_NAME}_feature"
FEATURE_TABLE_PATH = get_catalog_path(FEATURE_TABLE_NAME)
print(FEATURE_TABLE_PATH)

In [ ]:
if DATASET_PERSIST_MODE:
	feClient = FeatureEngineeringClient()
	feClient.create_table(
		name=FEATURE_TABLE_PATH,
		primary_keys=primary_key,
		df=feature_dbdf,
		description=f"{MANIFOLD_NAME} features (original)",
		tags={"source": "bronze", "format": "delta"}
	)

## Register representation model in Databricks

In [ ]:
MODEL_NAME = f"som-{MANIFOLD_NAME.lower()}"
MODEL_PATH = get_catalog_path(MODEL_NAME)
EXPERIMENT_NAME = f"/Users/matebalogh@ophelia-rnd.dev/Bianor_LilypondExperiments_{MANIFOLD_NAME}"
print(MODEL_PATH, EXPERIMENT_NAME)

In [ ]:
if MODEL_REGISTRATION_MODE:

	mlflow.set_experiment(experiment_name=EXPERIMENT_NAME)

	class MLflowSomModelWrapper(mlflow.pyfunc.PythonModel):
		from typing import Any

		def __init__(self, model:SomRepresentation, scaler):
			self.model = model
			self.scaler = scaler
		def predict(self, context, model_input, params: dict[str, Any] | None = None):
			"""First transforms the input data via scaler, then predicts the winner node of the SOM."""
			return [self.model.som.winner(x) for x in self.scaler.transform(model_input.to_numpy())]

	with mlflow.start_run():
		model = MLflowSomModelWrapper(som_rep, scaler)

		mlflow.log_metric("QE", som_rep.quantization_error)
		mlflow.log_metric("TE", som_rep.topographic_error)

		mlflow.pyfunc.log_model(
			python_model=model,
			name=MODEL_NAME,
			input_example=pd.DataFrame(X_scaled[:3]),
			pip_requirements=[
				"numpy",
				"pandas",
				"scikit-learn==1.5.2",
				"mlflow",
				"minisom",
			],
			registered_model_name=MODEL_PATH
		)

else: print("Skipping MLflow model registration.")

In [ ]:
version = 1

registered_model = f"{MODEL_NAME}/{version}"
print(registered_model)

registered_model_location = get_catalog_path(registered_model)
print(registered_model_location)

## Register metadata in Bianor

In [ ]:
from bianor_databricks_kit import BianorRecordManager
bianor_recorder = BianorRecordManager(catalog=CATALOG, schema=SCHEMA, spark=spark)

In [ ]:
if MODEL_REGISTRATION_MODE:

		# first registration
		# bianor_recorder.new_representation(name=f"{MANIFOLD_NAME} Manifold", som_model_location=registered_model_location, features_location="c.s.t")

		rep_id = "b228eb26-693d-4d26-9286-901e153109d2"

		# update existing
		bianor_recorder.update_representation(
			representation_id=rep_id,
			features_location=FEATURE_TABLE_PATH
		)

		# register projection layers

		import plotly.express as px

		sample_locations = [
			f"V_{MANIFOLD_NAME_SAFE}_low_manifold".lower(),
			f"V_{MANIFOLD_NAME_SAFE}_mid_manifold".lower(),
			f"V_{MANIFOLD_NAME_SAFE}_high_manifold".lower(),
		]
		colors = px.colors.sample_colorscale("Spectral_r", [0.0, 0.5, 1.0])
		names = [
			"Training data - Low manifold position",
			"Training data - Mid manifold position",
			"Training data - High manifold position"
		]

		for color, name, sample_loc in zip(colors, names, sample_locations):
			marker_dict = dict(
				color=color,
				symbol="star-diamond",
				opacity=0.9
			)

			proj_id = bianor_recorder.new_projection_layer(
				name=name,
				marker_dict=marker_dict,
				samples_location=get_catalog_path(sample_loc)
			)

			bianor_recorder.new_map_representation_projection_layer(representation_id=rep_id, projection_layer_id=proj_id)


else: print("Skipping Bianor metadata registration.")